### Imports

In [11]:
import datasets
import os
from itertools import islice
from dotenv import load_dotenv
load_dotenv()
datasets.logging.set_verbosity_error()
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from common.helper import load_hf_data, parse_price
categories = ["All_Beauty"]

### Database Connection

In [12]:
from cassandra.cluster import Cluster, ExecutionProfile, EXEC_PROFILE_DEFAULT
from cassandra.policies import DCAwareRoundRobinPolicy, TokenAwarePolicy
from cassandra.auth import PlainTextAuthProvider
from cassandra.io.asyncorereactor import AsyncoreConnection

def getCluster():
    profile = ExecutionProfile(load_balancing_policy=TokenAwarePolicy(DCAwareRoundRobinPolicy(local_dc='AWS_EU_WEST_1')))

    return Cluster(
        execution_profiles={EXEC_PROFILE_DEFAULT: profile},
        connection_class= AsyncoreConnection,
        contact_points=[
            "node-0.aws-eu-west-1.7bc61e2b60488a2ae30d.clusters.scylla.cloud", "node-1.aws-eu-west-1.7bc61e2b60488a2ae30d.clusters.scylla.cloud", "node-2.aws-eu-west-1.7bc61e2b60488a2ae30d.clusters.scylla.cloud"
        ],
        port=9042,
        auth_provider = PlainTextAuthProvider(username='scylla', password= os.environ['SCYLLA_PASSWORD']))

In [13]:
print('Connecting to cluster')
cluster = getCluster()
session = cluster.connect()

print('Connected to cluster %s' % cluster.metadata.cluster_name)

Connecting to cluster
Connected to cluster 85b700a6-1b4e-4fb9-a293-ffbb39de933d


In [14]:
print('Getting metadata')
for host in cluster.metadata.all_hosts():
    print('Datacenter: %s; Host: %s; Rack: %s' % (host.datacenter, host.address, host.rack))

Getting metadata
Datacenter: AWS_EU_WEST_1; Host: 52.51.186.102; Rack: euw1-az3
Datacenter: AWS_EU_WEST_1; Host: 52.211.101.68; Rack: euw1-az1
Datacenter: AWS_EU_WEST_1; Host: 63.34.122.250; Rack: euw1-az2


In [15]:
# 1. Create the Keyspace (The 'Database')
session.execute("""
    CREATE KEYSPACE IF NOT EXISTS amazon_data 
    WITH replication = {'class': 'NetworkTopologyStrategy', 'replication_factor': 1}
""")

### Creating Products Table

In [16]:
# Create the Products Table
# parent_asin is the unique ID we use to look up products
# Define the table using ScyllaDB/CQL types
session.execute(f"""
    CREATE TABLE IF NOT EXISTS amazon_data.raw_products (
        parent_asin text PRIMARY KEY,
        main_category text,
        title text,
        average_rating float,
        rating_number int,
        features list<text>,
        description list<text>,
        price float,
        images map<text,frozen <list<text>>>,
        videos map<text,frozen <list<text>>>,
        store text,
        categories list<text>,
        details text,
        bought_together text,
        subtitle text,
        author text   )
""")

print("ScyllaDB products table created successfully.")

ScyllaDB products table created successfully.


In [17]:
insert_query = session.prepare("""
    INSERT INTO amazon_data.raw_products (
        parent_asin,
        main_category,
        title,
        average_rating,
        rating_number,
        features,
        description,
        price,
        images,
        videos,
        store,
        categories,
        details,
        bought_together,
        subtitle,
        author
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")


#### Insert into Products Table

In [18]:
def insert_async(product_data):
    images = product_data.get("images") if isinstance(product_data.get("images"), dict) else None
    videos = product_data.get("videos") if isinstance(product_data.get("videos"), dict) else None
    params = [
        product_data.get('parent_asin'),
        product_data.get('main_category'),
        product_data.get('title'),
        product_data.get('average_rating'),
        product_data.get('rating_number'),
        product_data.get('features'),
        product_data.get('description'),
        parse_price(product_data.get('price')),
        images,
        videos,             
        product_data.get('store'),              
        product_data.get('categories'),         
        product_data.get("details"),       
        product_data.get('bought_together'),    
        product_data.get('subtitle'),           
        product_data.get('author')              
    ]
    return session.execute_async(insert_query, params)

In [19]:
import time
from tqdm import tqdm

print(("Insert Product Metadata"))
MAX_CONCURRENCY = 500
in_flight_requests = []
total_count = 0
start_time = time.perf_counter()

for category in categories:
    data = load_hf_data(
        root_split="raw",
        data_type="meta",
        category=category,
    )
    print(f"Processing category: {category}")

    for record in tqdm(data):
        total_count += 1

        # Fire async insert
        future = insert_async(record)
        in_flight_requests.append(future)

        # Manage concurrency
        if len(in_flight_requests) >= MAX_CONCURRENCY:
            for f in in_flight_requests:
                f.result()  # wait for all in-flight
            in_flight_requests = []

# Wait for any remaining in-flight requests
for f in in_flight_requests:
    f.result()

# Performance metrics
end_time = time.perf_counter()
duration = end_time - start_time
average_speed = total_count / duration if duration > 0 else 0

print("\nIngestion Complete")
print(f"Total Records: {total_count}")
print(f"Time Taken: {duration:.2f} seconds")
print(f"Average Throughput: {average_speed:.2f} records/sec")


Insert Product Metadata
Processing category: All_Beauty


112590it [06:27, 290.52it/s] 



Ingestion Complete
Total Records: 112590
Time Taken: 391.37 seconds
Average Throughput: 287.68 records/sec


In [21]:
row = session.execute("select count(*) from amazon_data.raw_products").one()
row

Row(count=112590)

## Creating Reviews Table

In [22]:
# Create the Reviews Table
session.execute("""
    CREATE TABLE IF NOT EXISTS amazon_data.raw_reviews (
        parent_asin text,
        user_id text,
        timestamp bigint,
        rating float,
        title text,
        text text,
        images list<frozen<map<text, text>>>,
        asin text,
        verified_purchase boolean,
        helpful_vote int,
        PRIMARY KEY (parent_asin, user_id,timestamp)
    ) WITH CLUSTERING ORDER BY (user_id ASC, timestamp DESC);
""")

#### Insert Statement for Reviews Table

In [30]:
insert_review_stmt = session.prepare("""
    INSERT INTO amazon_data.raw_reviews (
        parent_asin, user_id, timestamp, rating, title, 
        text, images, asin, verified_purchase, helpful_vote
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""")

def insert_review_async(review_data):
    params = [
        review_data.get('parent_asin'),
        review_data.get('user_id'),
        int(review_data.get('timestamp', 0)),
        float(review_data.get('rating', 0.0)),
        review_data.get('title'),
        review_data.get('text'),
        list(review_data.get('images', [])),
        review_data.get('asin'),
        bool(review_data.get('verified_purchase', False)),
        int(review_data.get('helpful_vote', 0))
    ]
    return session.execute_async(insert_review_stmt, params)

#### Insert into Reviews + Aggregated Reviews Table

In [31]:
import time
from tqdm import tqdm

MAX_CONCURRENCY = 500
in_flight_requests = []
total_count = 0
start_time = time.perf_counter()

for category in categories:
    data = load_hf_data(
        root_split="raw",
        data_type="review",
        category=category,
    )
    print(f"Processing category: {category}")

    for record in tqdm(data):
        total_count += 1

        # Fire async insert
        future_review = insert_review_async(record)

        in_flight_requests.append(future_review)
        

        # Manage concurrency
        if len(in_flight_requests) >= MAX_CONCURRENCY:
            for f in in_flight_requests:
                f.result()  # wait for all in-flight
            in_flight_requests = []

# Wait for any remaining in-flight requests
for f in in_flight_requests:
    f.result()

# Performance metrics
end_time = time.perf_counter()
duration = end_time - start_time
average_speed = total_count / duration if duration > 0 else 0

print("\nIngestion Complete")
print(f"Total Records: {total_count}")
print(f"Time Taken: {duration:.2f} seconds")
print(f"Average Throughput: {average_speed:.2f} records/sec")


Processing category: All_Beauty


701528it [09:47, 1194.95it/s]


Ingestion Complete
Total Records: 701528
Time Taken: 588.24 seconds
Average Throughput: 1192.60 records/sec


In [32]:
row = session.execute("select count(*) from amazon_data.raw_reviews")
row.one()

Row(count=694252)

## Task 1 Querying

##### 1.Product Information Retrieval: Given a product ID (parent_asin)the product title and full product description

In [34]:
import time
import random
import numpy as np

random_asins = ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W', 'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']

durations = []

# Step 2: Measure query performance
for asin in random_asins:
    query = "SELECT title, description FROM amazon_data.raw_products WHERE parent_asin=%s"
    times = []
    for _ in range(10):  # run 10 times per ASIN
        start = time.time()
        session.execute(query, (asin,))
        end = time.time()
        times.append((end - start) * 1000)  # ms
    durations.append(np.mean(times))

# Step 3: Print stats
print("Product Info Query (Cassandra):")
print(f"Median: {np.median(durations):.3f} ms")
print(f"Mean:   {np.mean(durations):.3f} ms")


Product Info Query (Cassandra):
Median: 30.623 ms
Mean:   31.294 ms


##### 2. Recent Product Reviews Given a product ID (parent_asin), retrieve asin, retrieve the most recent N customer reviews, including rating, review title, review text, and timestamp.

In [35]:
import time
import random
import numpy as np

N = 10          # number of recent reviews per product
# Pick 10 random ASINs
random_asins = ['B07484MS4G', 'B01FYDF3VI', 'B01C3PMM7S', 'B012HUTKTM', 'B075R6NQ1W', 'B08DCZGQJ5', 'B077WBYPLG', 'B001KYU8UM', 'B0919HCJDN', 'B08MQCXGJM']

durations = []

# Step 2: Measure query performance
for asin in random_asins:
    query = "SELECT rating, title, text, timestamp FROM amazon_data.raw_reviews WHERE parent_asin=%s LIMIT %s"
    times = []
    for _ in range(10):  # run 10 times per ASIN
        start = time.time()
        session.execute(query, (asin, N))
        end = time.time()
        times.append((end - start) * 1000)  # convert to ms
    durations.append(np.mean(times))

# Step 3: Print stats
print("Recent Reviews Query (Cassandra):")
print(f"Median: {np.median(durations):.3f} ms")
print(f"Mean:   {np.mean(durations):.3f} ms")


Recent Reviews Query (Cassandra):
Median: 30.933 ms
Mean:   31.188 ms


#### Query 4 User Review History: Given a user ID, retrieve all products reviewed by that user along with the corresponding ratings and review dates.

Since ScyllaDb is query first we create another table for the query making the USER_ID the primary key

In [36]:
session.execute("""CREATE TABLE amazon_data.user_reviews_query (
    user_id text,
    parent_asin text,
    rating float,
    timestamp bigint,
    title text,
    text text,
    PRIMARY KEY (user_id, timestamp, parent_asin)
) WITH CLUSTERING ORDER BY (timestamp DESC,parent_asin DESC);
""")

Data Ingestion

In [37]:
# Prepare the insert statement
insert_user_review_stmt = session.prepare("""
    INSERT INTO amazon_data.user_reviews_query (
        user_id, timestamp, parent_asin, rating, title, text
    ) VALUES (?, ?, ?, ?, ?, ?)
""")

# Async insert helper
def insert_user_review_async(review_data):
    params = [
        review_data.get('user_id'),
        int(review_data.get('timestamp', 0)),
        review_data.get('parent_asin'),
        float(review_data.get('rating', 0.0)),
        review_data.get('title'),
        review_data.get('text'),
    ]
    return session.execute_async(insert_user_review_stmt, params)

#### Insertion into the Query Table for User Reviews for query 4

In [38]:
import time
from tqdm import tqdm

MAX_CONCURRENCY = 500
in_flight_requests = []
total_count = 0
start_time = time.perf_counter()

for category in categories:
    data = load_hf_data(
        root_split="raw",
        data_type="review",
        category=category,
    )
    print(f"Processing category: {category}")

    for record in tqdm(data):
        total_count += 1

        # Fire async insert
        future = insert_user_review_async(record)
        in_flight_requests.append(future)

        # Manage concurrency
        if len(in_flight_requests) >= MAX_CONCURRENCY:
            for f in in_flight_requests:
                f.result()  # wait for all in-flight
            in_flight_requests = []

# Wait for any remaining in-flight requests
for f in in_flight_requests:
    f.result()

# Performance metrics
end_time = time.perf_counter()
duration = end_time - start_time
average_speed = total_count / duration if duration > 0 else 0

print("\nIngestion Complete")
print(f"Total Records: {total_count}")
print(f"Time Taken: {duration:.2f} seconds")
print(f"Average Throughput: {average_speed:.2f} records/sec")


Processing category: All_Beauty


701528it [08:44, 1337.28it/s]


Ingestion Complete
Total Records: 701528
Time Taken: 525.73 seconds
Average Throughput: 1334.39 records/sec


In [39]:
import time
import random
import numpy as np


# Pick 10 random users
random_users = ['AG5JJZKUDTC5PREHPTI5NOC75M2Q', 'AFQIHDRZZ7RGRLSFKDGCYPBHHTKA', 'AFXCPSCWLOSVQZHN7FAECOYAT3RA', 'AHT6VS3G3JSA7KBOCBT6KPUCJX7A', 'AH4G4SNKSIWZNGZ5EDNJTRORKIMQ', 'AG4JLUSDCYTYSSE5SQADXOPATLTA', 'AGXVAAK6XJ7C2GUTYQEMBOGQ7JGQ', 'AGKPEXI6SAACYVXRNMICW3NKG2XA', 'AHA272AOVKWZUXL672G653A2MTIA', 'AH6X5VUV5AUGO3G55G7AL6GZ2QZQ']

durations = []

# Step 2: Measure query performance
for user_id in random_users:
    query = """
    SELECT parent_asin, rating, timestamp, title, text
    FROM amazon_data.user_reviews_query
    WHERE user_id=%s
    """
    
    times = []
    for _ in range(10):
        start = time.time()
        session.execute(query, (user_id,))
        end = time.time()
        times.append((end - start) * 1000)  # convert to ms
    
    durations.append(np.mean(times))

# Step 3: Print stats
print("User Reviews Query (Cassandra):")
print(f"Median: {np.median(durations):.3f} ms")
print(f"Mean:   {np.mean(durations):.3f} ms")


User Reviews Query (Cassandra):
Median: 60.031 ms
Mean:   58.658 ms


### Query 5: Retrieve the average rating, rating distribution (count per star level), and total number of reviews for a given product.

In [ ]:
query = """
    SELECT rating
    FROM amazon_data.raw_reviews
    WHERE parent_asin=%s
"""

rows = session.execute(query, ('B07484MS4G',))

rating_counts = {1:0, 2:0, 3:0, 4:0, 5:0}
total_reviews = 0
rating_sum = 0

for row in rows:
    r = int(row.rating)
    rating_counts[r] += 1
    total_reviews += 1
    rating_sum += r

avg_rating = rating_sum / total_reviews if total_reviews > 0 else 0

print("Total Reviews:", total_reviews)
print("Average Rating:", avg_rating)
print("Rating Counts:", rating_counts)

InvalidRequest: Error from server: code=2200 [Invalid query] message="Group by non-primary-key column rating"